In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #Genera datos (imagenes) sintéticos
from keras import optimizers #Técnica del descenso del gradiente, sirve para minimizar el error en el proceso de entrenamiento
from keras.models import Sequential # Estacebe un modelo de red neuronal por capas
from keras.layers import Dense, Flatten, Dropout, Activation
#Dense = define las neuronas por cada capa
#Flatter = aplana los datos en formato matriz N-Tensor --> 1-Tensor
#Dropout = técnica de reducción del sobreajuste o sobre entrenamiento (apagar % de neuronas)
#Activation = determina las funciones de activiación en cada neurona (Relu, sigmoid, softmax, tanh, linear, etc)
from keras.layers import Convolution2D, MaxPooling2D


In [1]:
# APARTIR DE AQUÍ SE MANEJARA LO NECESARIO PARA LA IMPLEMENTACIÓN DE LA INTERFAZ GRAFICA.
#PILLOW: PROCESAMIENTO DE IMAGENES 

#ABAJO SE IMPORTARÁ TODO LO NECESARIO
import tkinter as tk  # IMPORTAMOS TKINTER PARA CREAR INTERFACES GRÁFICAS (VENTANAS, BOTONES, ETC.)
from tkinter import ttk, filedialog, messagebox, scrolledtext  # IMPORTAMOS COMPONENTES AVANZADOS DE TKINTER: ESTILOS, DIÁLOGOS Y ÁREAS DE TEXTO DESPLAZABLES

import threading  # IMPORTAMOS THREADING PARA EJECUTAR PROCESOS EN SEGUNDO PLANO SIN CONGELAR LA INTERFAZ
import queue  # IMPORTAMOS QUEUE PARA MANEJAR COLAS DE MENSAJES ENTRE HILOS (THREADS)
import os  # IMPORTAMOS OS PARA INTERACTUAR CON EL SISTEMA OPERATIVO (ARCHIVOS, RUTAS, ETC.)
import time  # IMPORTAMOS TIME PARA MANEJAR TIEMPOS, PAUSAS Y MEDICIONES

from concurrent.futures import ThreadPoolExecutor  # IMPORTAMOS THREADPOOL PARA EJECUTAR TAREAS CON HILOS DE FORMA EFICIENTE
from datetime import datetime  # IMPORTAMOS DATETIME PARA MANEJO DE FECHAS Y HORAS

from dataclasses import dataclass  # IMPORTAMOS DATACLASS PARA CREAR CLASES SIMPLES PARA ALMACENAR DATOS
from enum import Enum, auto  # IMPORTAMOS ENUM PARA CREAR ENUMERACIONES (CONSTANTES AGRUPADAS)

from typing import Optional, Callable  # IMPORTAMOS TYPING PARA DEFINIR TIPOS OPCIONALES Y FUNCIONES COMO PARÁMETROS

import sys  # IMPORTAMOS SYS PARA ACCEDER A FUNCIONES DEL SISTEMA Y PARÁMETROS DEL INTÉRPRETE DE PYTHON

In [ ]:
# /*APARTIR DE AQUÍ SE CONSTRUIRÁ todo LO RELACIONADO A LA INTERFAZ*/


APARTIR DE ACA ABAJO DE CONSTRUIRÁ LA RED NEURONAL

In [3]:
#Definir los hiperparámetros de la red neurona convolucional (Convolutional Neural Network - CNN)

#Definir la ruta de los datos de entranamiento
entrenar = "CNN_Imagenes/entrenar"
validar = "CNN_Imagenes/validar"

#Hiperparámetros
epocas = 20
altura,anchura = 200,200
batch_size = 2
pasos = 100

#Definir la cantidad de kernels por cada capa
kernel1=32 # 2,4,8,16,32,64,128,256, 512, etc
kernel1_size = (3,3)
kernel2=64
kernel2_size = (4,4)
size_pooling = (3,3)
clases = 2 #Número de objetos a detectar o identificar




In [5]:
#Generar datos sintéticos (se recomienda si la cantidad de datos es pequeña)

entrenamiento = ImageDataGenerator(rescale=1/255,
                            zoom_range=0.2,
                            horizontal_flip=True)

validacion = ImageDataGenerator(rescale=1/255)

#Extraer las imagenes de las carpetas

imagenes_entrenamiento = entrenamiento.flow_from_directory(entrenar,
                                                    target_size=(anchura,altura),
                                                    batch_size=batch_size,
                                                    class_mode="categorical")
imagenes_validacion = validacion.flow_from_directory(validar,
                                                target_size=(anchura,altura),
                                                batch_size=batch_size,
                                                class_mode="categorical")

Found 12 images belonging to 2 classes.
Found 8 images belonging to 2 classes.


In [6]:
#Definir la arquitectura de la red neuronal convolucional

CNN = Sequential()

CNN.add(Convolution2D(kernel1,
                    kernel1_size,
                    padding="same",
                    input_shape=(altura,anchura,3),
                    activation="relu")) #Primera capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

CNN.add(Convolution2D(kernel2,
                    kernel2_size,
                    padding="same",
                    input_shape=(altura,anchura,3),
                    activation="relu")) #Segunda capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

#Aplanar las matrices en formato de vector
CNN.add(Flatten())

#Conectar a el perceptrón multicapa
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dropout(0.5))

#Definir la capa de salida
CNN.add(Dense(clases,activation="softmax"))


c:\Users\alexi\OneDrive\Desktop\SII_CNN\CNN_Agua\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
#Definir los parámetros del entrenamiento
CNN.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["acc","mse"])

In [ ]:
#Realizamos el entrenamiento
historico = CNN.fit(imagenes_entrenamiento,
                    validation_data=imagenes_validacion,
                    epochs=epocas,
                    validation_steps=pasos,
                    verbose=1)

In [ ]:
#Guarda el modelo entrenado
CNN.save("CNN_Imagenes/Modelo/cnn.h5")
CNN.save_weights("CNN_Imagenes/Modelo/cnn_pesos.weights.h5")

In [9]:
#Evaluando el modelo entrenado
#AQUÍ SE EVALUARÁ

import numpy as np
import pillow_avif
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os.path

imagen = "CNN_Imagenes/validar/perro/perro7.jpg"

altura,anchura = 200,200
modelo = "CNN_Imagenes/Modelo/cnn.h5"
pesos = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"

#Cargar la arquitectura de la cnn y sus pesos

cnn = load_model(modelo)
cnn.load_weights(pesos)

#Clasificar la imagen o el objeto

imagen_clasificar = load_img(imagen,target_size=(anchura,altura))
imagen_clasificar = img_to_array(imagen_clasificar)
imagen_clasificar = np.expand_dims(imagen_clasificar, axis=0)


#Evaluar

clase = cnn.predict(imagen_clasificar)

arg_max = np.argmax(clase[0])

if (arg_max ==0):
    print("gallo")
elif(arg_max==1):
    print("perro")    


c:\Users\alexi\OneDrive\Desktop\SII_CNN\CNN_Agua\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 26 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step
perro


In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os.path
import cv2

def evaluar(imagen):
    altura,anchura = 200,200
    modelo = "CNN_Imagenes/Modelo/cnn.h5"
    pesos = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"

    #Cargar la arquitectura de la cnn y sus pesos

    cnn = load_model(modelo)
    cnn.load_weights(pesos)

    #Clasificar la imagen o el objeto

    imagen_clasificar = cv2.resize(imagen,(anchura,altura))
    imagen_clasificar = imagen_clasificar/255
    
    imagen_clasificar = img_to_array(imagen_clasificar)
    imagen_clasificar = np.expand_dims(imagen_clasificar, axis=0)


    #Evaluar

    clase = cnn.predict(imagen_clasificar)

    arg_max = np.argmax(clase[0])

    if (arg_max ==0):
        print("gallo")
    elif(arg_max==1):
        print("perro")    


: 